In [15]:
import html_processor
import project_name_tagger
import resource_table_extractor
import table_processor
import classifier
importlib.reload(html_processor)
importlib.reload(project_name_tagger)
importlib.reload(resource_table_extractor)
importlib.reload(table_processor)
importlib.reload(classifier)

{'category': {'indicated': {'indicated', 'i', 'ind'},
              'inferred': {'inf', 'inferred', 'infe'},
              'meas_ind': {'m & i',
                           'm + i',
                           'm / i',
                           'm&i',
                           'm+i',
                           'm/i',
                           'meas & ind',
                           'meas / ind',
                           'meas ind',
                           'meas&ind',
                           'meas-ind',
                           'meas/ind',
                           'meas_ind',
                           'measured & indicated',
                           'measured and indicated',
                           'mi'},
              'measured': {'m', 'meas', 'measured'},
              'probable': {'prob', 'probable', 'proba'},
              'prov_prob': {'p & p',
                            'p + p',
                            'p / p',
                            'p&p',
          

<module 'classifier' from '/home/ravi/work/mining/project/classifier/__init__.py'>

In [86]:
# Set maximum number of rows to display
pd.set_option('display.max_rows', 2000)

# Set maximum number of columns to display
pd.set_option('display.max_columns', 2000)

# Set maximum width of each column
pd.set_option('display.max_colwidth', 2000)

# Set the width of the display in characters
pd.set_option('display.width', 2200)


In [9]:
import pandas as pd
from tqdm.notebook import tqdm

import importlib

#### Search for mining news articles (8000 urls)

In [ ]:
from search import search_news

urls = search_news(8000)
df = pd.DataFrame(urls, columns=['url'])
df.head()

#### Download the html files

In [19]:
from download import fetch_and_write_all

file_paths = await fetch_and_write_all(urls)
df['path'] = file_paths
df.head()

,url,path
0,http://www.newswire.ca/en/releases/archive/Feb...,articles/February202623c1544.html
1,http://www.newswire.ca/en/releases/archive/Feb...,articles/February202623c5893.html
2,http://www.newswire.ca/en/releases/archive/Feb...,articles/February202623c4520.html
3,http://www.newswire.ca/en/releases/archive/Feb...,articles/February202623c4355.html
4,http://www.newswire.ca/en/releases/archive/Feb...,articles/February202623c1563.html


In [21]:
df.index.name = 'id'
df.to_csv("data.csv")

In [10]:
df = pd.read_csv("data.csv")
df.set_index('id', inplace=True)

#### Tag articles with mentioned project / mine names

In [ ]:
import html_processor
import project_name_tagger
importlib.reload(html_processor)
importlib.reload(project_name_tagger)

from html_processor import get_article
from project_name_tagger import get_project_names
import time

def p_name_helper(title, provider, published_date, body):
    try:
        project_names = get_project_names(f"TITLE: {title}, PROVIDER: {provider}, PUBLISHED_DATE: {published_date}, BODY: {body}")
        return str(project_names)
    except:
        return None

times = 0
total = len(df)
processed_count = 0

if 'project_names' not in df.columns:
    df['project_names'] = ''

for i, [idx, row] in tqdm(enumerate(df.iterrows()), total=total):
    start = time.time()

    if i < 1900:
        continue

    existing_projects = df.at[idx, 'project_names']
    if existing_projects is not None and pd.notna(existing_projects) and existing_projects != '[]' and existing_projects != '':
        # print('exists')
        continue
        
    try:
        with open(row.get('path'), 'r') as file:
            title, provider, published_date, body = get_article(file.read(), redact_tables=True, convert_to_text=True)
    except:
        # print('exception')
        continue

    if len(body) / 3 > 10000:
        # print('too long')
        continue

    result = p_name_helper(title, provider, published_date, body)
    df.at[idx, 'project_names'] = result
    
    if i%50 == 0:
        df.to_csv("data.csv")

    processed_count += 1
        
    end = time.time()
    times += end - start
    avg_time = times / processed_count

    # print(f"Processed {processed_count} docs, Average time per doc: {avg_time:.4f} seconds", end='\n')
    print(f"Processed {processed_count} docs, Average time per doc: {avg_time:.4f} seconds, estimated time: {(total - i) * avg_time / 60 / 60:.4f} hours", end='\r')

  0%|          | 0/8000 [00:00<?, ?it/s]

Processed 3756 docs, Average time per doc: 7.2110 seconds, estimated time: 4.4628 hourss

In [7]:
df.describe()

,url,path,project_names
count,8000,7968,5589
unique,7999,7967,4814
top,http://www.newswire.ca/en/releases/archive/Jul...,articles/July202508c9439.html,"['Lunahuasi', 'Los Helados']"
freq,2,2,26


#### Classification of articles into Resource, Drilling, Production
Only 'Resource' classification is implemented

In [3]:
from classifier import is_resource_table

if not 'category' in df.columns:
    df['category'] = ''

for idx, row in tqdm(df.iterrows(), total=len(df)):
    try:
        tables = pd.read_html(row.get('path'))
    except:
        tables = []
    for table in tables:
        is_resource = is_resource_table(table)
        df.at[idx, 'category'] = 'resource'
        break;

['ton', 'cont', 'tons', 'tonn', 'metal', 'tonnes', 'average', 'tonnage', 'content', 'contained', 'avg', 'grade']
['proven', 'meas / ind', 'infe', 'm+i', 'p', 'prov_prob', 'meas', 'pro', 'proven & probable', 'p & p', 'p/p', 'prov-prob', 'prov / prob', 'meas/ind', 'm & i', 'prov prob', 'inf', 'meas-ind', 'i', 'p / p', 'proven / probable', 'measured', 'proven + probable', 'm / i', 'ind', 'meas&ind', 'indicated', 'meas ind', 'prov', 'prob', 'meas_ind', 'p + p', 'inferred', 'm/i', 'meas & ind', 'm&i', 'proven/probable', 'mi', 'm + i', 'prov/prob', 'probable', 'm', 'proba']


  0%|          | 0/8000 [00:00<?, ?it/s]

In [20]:
df.describe()

,url,path,project_names,category
count,8000,7968,5589,2383
unique,7999,7967,4814,1
top,http://www.newswire.ca/en/releases/archive/Jul...,articles/July202508c9439.html,"['Lunahuasi', 'Los Helados']",resource
freq,2,2,26,2383


In [21]:
df.to_csv('data.csv')

In [26]:


import sys
sys.path.append("..") # Adds higher directory to python modules path.


In [70]:
tables = pd.read_html('articles/February202623c6856.html')
tables[0]

,0,1,2,3,4
0,Mineral Resource Estimate,Mineral Resource Estimate,Mineral Resource Estimate,Mineral Resource Estimate,Mineral Resource Estimate
1,Domain,Tonnes Above Cut- off (millions),Li Grade (ppm),Li Contained (million t),LCE (million t)
2,Measured,858.26,990,0.850,4.523
3,Indicated,280.33,891,0.250,1.329
4,Measured & Indicated,1138.59,966,1.099,5.852
5,Inferred,187.28,820,0.154,0.817
6,"1.The effective date of the Mineral Resource Estimate is April 29, 2024. The QP for the estimate...","1.The effective date of the Mineral Resource Estimate is April 29, 2024. The QP for the estimate...","1.The effective date of the Mineral Resource Estimate is April 29, 2024. The QP for the estimate...","1.The effective date of the Mineral Resource Estimate is April 29, 2024. The QP for the estimate...","1.The effective date of the Mineral Resource Estimate is April 29, 2024. The QP for the estimate..."


In [100]:
from resource_table_extractor import parse_table
from classifier import is_resource_table

for idx, row in df.iterrows():
    try:
        tables = pd.read_html(row.get('path'))
    except:
        tables = []
    for i, table in enumerate(tables):
        is_resource = is_resource_table(table)
        if is_resource:
            parsed_table = parse_table(table)
            if parsed_table is None:
                continue
            print("file:///home/ravi/work/mining/project/"+row.get('path'))
            print(i+1)
            display(parsed_table)
            display(HTML(table.to_html()))
            

file:///home/ravi/work/mining/project/articles/February202623c6856.html
1


,value,metric,category,measurement,commodity
0,858.26,tonnage,measured,NaN,NaN
1,990,grade,measured,ppm,lithium
2,0.850,contained,measured,t,lithium
3,4.523,NaN,measured,t,lithium (lce)
4,280.33,tonnage,indicated,NaN,NaN
5,891,grade,indicated,ppm,lithium
6,0.250,contained,indicated,t,lithium
7,1.329,NaN,indicated,t,lithium (lce)
8,1138.59,tonnage,meas_ind,NaN,NaN
9,966,grade,meas_ind,ppm,lithium


,0,1,2,3,4
0,Mineral Resource Estimate,Mineral Resource Estimate,Mineral Resource Estimate,Mineral Resource Estimate,Mineral Resource Estimate
1,Domain,Tonnes Above Cut- off (millions),Li Grade (ppm),Li Contained (million t),LCE (million t)
2,Measured,858.26,990,0.850,4.523
3,Indicated,280.33,891,0.250,1.329
4,Measured & Indicated,1138.59,966,1.099,5.852
5,Inferred,187.28,820,0.154,0.817
6,"1.The effective date of the Mineral Resource Estimate is April 29, 2024. The QP for the estimate is Ms. Terre Lane, MMSA, an employee of GRE and independent of Century.2.The Mineral Resources are constrained by a pit shell with a 200 ppm Li cut-off and density of 1.505 g/cm3. The cut-off grade considers an operating cost of$20/t mill feed, process recovery of 78% and a long-term lithium carbonate price of $24,000/t.3.The Mineral Resource estimate was prepared in accordance with 2014 CIM Definition Standards and the 2019 CIM Best Practice Guidelines.4.Mineral Resource figures have been rounded.5.One tonne of lithium = 5.323 tonnes lithium carbonate.6.Mineral Resources are inclusive of Mineral Reserves.","1.The effective date of the Mineral Resource Estimate is April 29, 2024. The QP for the estimate is Ms. Terre Lane, MMSA, an employee of GRE and independent of Century.2.The Mineral Resources are constrained by a pit shell with a 200 ppm Li cut-off and density of 1.505 g/cm3. The cut-off grade considers an operating cost of$20/t mill feed, process recovery of 78% and a long-term lithium carbonate price of $24,000/t.3.The Mineral Resource estimate was prepared in accordance with 2014 CIM Definition Standards and the 2019 CIM Best Practice Guidelines.4.Mineral Resource figures have been rounded.5.One tonne of lithium = 5.323 tonnes lithium carbonate.6.Mineral Resources are inclusive of Mineral Reserves.","1.The effective date of the Mineral Resource Estimate is April 29, 2024. The QP for the estimate is Ms. Terre Lane, MMSA, an employee of GRE and independent of Century.2.The Mineral Resources are constrained by a pit shell with a 200 ppm Li cut-off and density of 1.505 g/cm3. The cut-off grade considers an operating cost of$20/t mill feed, process recovery of 78% and a long-term lithium carbonate price of $24,000/t.3.The Mineral Resource estimate was prepared in accordance with 2014 CIM Definition Standards and the 2019 CIM Best Practice Guidelines.4.Mineral Resource figures have been rounded.5.One tonne of lithium = 5.323 tonnes lithium carbonate.6.Mineral Resources are inclusive of Mineral Reserves.","1.The effective date of the Mineral Resource Estimate is April 29, 2024. The QP for the estimate is Ms. Terre Lane, MMSA, an employee of GRE and independent of Century.2.The Mineral Resources are constrained by a pit shell with a 200 ppm Li cut-off and density of 1.505 g/cm3. The cut-off grade considers an operating cost of$20/t mill feed, process recovery of 78% and a long-term lithium carbonate price of $24,000/t.3.The Mineral Resource estimate was prepared in accordance with 2014 CIM Definition Standards and the 2019 CIM Best Practice Guidelines.4.Mineral Resource figures have been rounded.5.One tonne of lithium = 5.323 tonnes lithium carbonate.6.Mineral Resources are inclusive of Mineral Reserves.","1.The effective date of the Mineral Resource Estimate is April 29, 2024. The QP for the estimate is Ms. Terre Lane, MMSA, an employee of GRE and independent of Century.2.The Mineral Resources are constrained by a pit shell with a 200 ppm Li cut-off and density of 1.505 g/cm3. The cut-off grade considers an operating cost of$20/t mill feed, process recovery of 78% and a long-term lithium carbonate price of $24,000/t.3.The Mineral Resource estimate was prepared in accordance with 2014 CIM Definition Standards and the 2019 CIM Best Practice Guidelines.4.Mineral Resource figures have been rounded.5.One tonne of lithium = 5.323 tonnes lithium carbonate.6.Mineral Resources are inclusive of Mineral Reserves."


file:///home/ravi/work/mining/project/articles/February202623c6856.html
2


,value,metric,category,measurement,commodity
0,266.39,tonnage,proven,NaN,NaN
1,1147,grade,proven,ppm,lithium
2,0.306,contained,proven,t,lithium
3,1.626,NaN,proven,t,lithium (lce)
4,21.26,tonnage,probable,NaN,NaN
5,1174,grade,probable,ppm,lithium
6,0.025,contained,probable,t,lithium
7,0.133,NaN,probable,t,lithium (lce)
8,287.65,tonnage,prov_prob,NaN,NaN
9,1149,grade,prov_prob,ppm,lithium


...

,0,1,2,3,4,5,6,7,8
0,(1),(2),(2),(3),NaN,(4),(4),(5),(5)
1,Hole-ID,Grade Control Program,Grade Control Program,Block Model,NaN,Grade Control Program,Grade Control Program,Block Model,Block Model
2,Hole-ID,Total Length (m),Au Capped (g/t),Au (g/t),NaN,Length(m),Au Capped (g/t),Length(m),Au (g/t)
3,NaN,Au > 0.0 g/t,Au > 0.0 g/t,Au > 0.0 g/t,NaN,Au >= 0.80,Au >= 0.80,Au >= 0.80,Au >= 0.80
4,MRE-FG25-001,63,0.93,1.14,NaN,24,1.78,39,1.53
5,MRE-FG25-002,70,1.42,1.23,NaN,37,2.21,52,1.49
6,MRE-FG25-003,64,1.54,1.07,NaN,43,2.06,41,1.35
7,MRE-FG25-004,66,1.02,1.18,NaN,41,1.42,41,1.65
8,MRE-FG25-005,64,1.17,2.22,NaN,35,2.06,51,2.63
9,MRE-FG25-006,64,0.92,1.23,NaN,15,2.22,46,1.58


file:///home/ravi/work/mining/project/articles/February202610c0112.html
17


,value,category,commodity,minetype,measurement,metric
0,3516000,indicated,gold,underground,g/t,tonnage
1,4.41,indicated,gold,underground,g/t,grade
2,499000,indicated,gold,underground,g/t,contained
3,5490000,inferred,gold,underground,g/t,tonnage
4,3.65,inferred,gold,underground,g/t,grade
5,644000,inferred,gold,underground,g/t,contained


,0,1,2,3
0,Table 9: True North Gold Project: Underground Mineral Resource Estimate,Table 9: True North Gold Project: Underground Mineral Resource Estimate,Table 9: True North Gold Project: Underground Mineral Resource Estimate,Table 9: True North Gold Project: Underground Mineral Resource Estimate
1,(within 2.25 g/t Au mineral resource constraining envelopes),(within 2.25 g/t Au mineral resource constraining envelopes),(within 2.25 g/t Au mineral resource constraining envelopes),(within 2.25 g/t Au mineral resource constraining envelopes)
2,Mineral Resource,Tonnage,Gold Grade,Contained Gold
3,(Category),(t),(g/t),(oz)
4,Indicated Resources,3516000,4.41,499000
5,Inferred Resources,5490000,3.65,644000


KeyError: ['metric', 'category']

In [101]:
test = pd.read_html("/home/ravi/work/mining/project/articles/February202618c1164.html")

In [107]:
test[4].to_html()

'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th></th>\n      <th>0</th>\n      <th>1</th>\n      <th>2</th>\n      <th>3</th>\n      <th>4</th>\n      <th>5</th>\n      <th>6</th>\n      <th>7</th>\n      <th>8</th>\n      <th>9</th>\n      <th>10</th>\n      <th>11</th>\n      <th>12</th>\n      <th>13</th>\n      <th>14</th>\n      <th>15</th>\n      <th>16</th>\n      <th>17</th>\n      <th>18</th>\n      <th>19</th>\n      <th>20</th>\n      <th>21</th>\n      <th>22</th>\n      <th>23</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <th>0</th>\n      <td>Mineral Resources (100% basis,\xa0inclusive of Reserves)</td>\n      <td>Mineral Resources (100% basis,\xa0inclusive of Reserves)</td>\n      <td>Mineral Resources (100% basis,\xa0inclusive of Reserves)</td>\n      <td>Mineral Resources (100% basis,\xa0inclusive of Reserves)</td>\n      <td>Mineral Resources (100% basis,\xa0inclusive of Reserves)</td>\n      <td>Mineral Resou

In [ ]:
import importlib
from project_name_tagger import train_ner
importlib.reload(train_ner)
from project_name_tagger.train_ner import load_data


data = load_data('data.csv')

In [11]:
import pickle

with open("ner_train_data.pkl", 'wb') as f:
    pickle.dump(data, f)

In [1]:
import importlib
from project_name_tagger import train_ner
importlib.reload(train_ner)
from project_name_tagger.train_ner import train_project_ner_sm
import pickle
import spacy
spacy.require_cpu()

with open("ner_train_data.pkl", 'rb') as f:
    data = pickle.load(f)

model = train_project_ner_sm(data)

Train size: 4264, Dev size: 1066


/home/ravi/work/mining/project/.venvgpu/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/ravi/work/mining/project/.venvgpu/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Iter 1/20 Losses: {'tok2vec': 0.0, 'tagger': 0.0, 'parser': 0.0, 'ner': np.float32(1543266.2)}


/home/ravi/work/mining/project/.venvgpu/lib/python3.10/site-packages/spacy/pipeline/attributeruler.py:137: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  matches = self.matcher(doc, allow_missing=True, as_spans=False)
/home/ravi/work/mining/project/.venvgpu/lib/python3.10/site-packages/spacy/pipeline/lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


Validation (dev) score at epoch 1: 0.0 F-score
Iter 2/20 Losses: {'tok2vec': 0.0, 'tagger': 0.0, 'parser': 0.0, 'ner': np.float32(77860.07)}
Validation (dev) score at epoch 2: 0.0 F-score
Iter 3/20 Losses: {'tok2vec': 0.0, 'tagger': 0.0, 'parser': 0.0, 'ner': np.float32(73146.45)}
Validation (dev) score at epoch 3: 0.0 F-score
Iter 4/20 Losses: {'tok2vec': 0.0, 'tagger': 0.0, 'parser': 0.0, 'ner': np.float32(70974.28)}
Validation (dev) score at epoch 4: 0.0 F-score
Iter 5/20 Losses: {'tok2vec': 0.0, 'tagger': 0.0, 'parser': 0.0, 'ner': np.float32(67928.76)}
Validation (dev) score at epoch 5: 0.0009432182607055273 F-score
Iter 6/20 Losses: {'tok2vec': 0.0, 'tagger': 0.0, 'parser': 0.0, 'ner': np.float32(66515.96)}
Validation (dev) score at epoch 6: 0.019097061277463613 F-score
Iter 7/20 Losses: {'tok2vec': 0.0, 'tagger': 0.0, 'parser': 0.0, 'ner': np.float32(64710.176)}
Validation (dev) score at epoch 7: 0.05141388174807198 F-score
Iter 8/20 Losses: {'tok2vec': 0.0, 'tagger': 0.0, 'pars

In [ ]:
# import spacy
nlp = spacy.load("model_project_only")

In [1]:
from html_processor import get_article

try:
    with open("articles/July202508c9439.html", "r") as file:
        title, provider, published_date, body = get_article(
            file.read(), redact_tables=True, convert_to_text=True
        )
except:
    pass
text = f"TITLE: {title}, PROVIDER: {provider}, PUBLISHED_DATE: {published_date}, BODY: {body}"
doc = nlp(text)
doc.ents

NameError: name 'nlp' is not defined

In [45]:
body = f"""SilverRock Mining Ltd. has released an updated mineral resource estimate for its flagship Eagle Ridge Project, located in the heart of the Iron Valley mining district. The new estimate, prepared by independent consultants, highlights a substantial increase in both indicated and inferred resources.

According to the report, the Eagle Ridge deposit now contains an estimated 2.5 million ounces of gold equivalent in the indicated category and an additional 1.1 million ounces in the inferred category. This marks a 30% increase from the previous estimate released last year.

“We are extremely encouraged by these results,” said SilverRock CEO, Maria Thompson. “The updated resource estimate confirms Eagle Ridge as one of the most promising undeveloped gold projects in the region. Our ongoing drilling program continues to intersect high-grade mineralization, and we anticipate further growth in the resource base.”

The resource estimate is based on over 120,000 meters of drilling completed to date. The company plans to continue exploration at Eagle Ridge, with a focus on expanding the high-grade zones and testing new targets within the 15,000-hectare property.

SilverRock Mining also holds interests in the nearby Granite Peak and Willow Creek projectskj, both of which are undergoing early-stage exploration.

The company will host a conference call with investors and analysts next week to discuss the results and outline next steps for the Eagle Ridge Project.

"""
text = f"TITLE: {""}, PROVIDER: {""}, PUBLISHED_DATE: {""}, BODY: {body}"

In [46]:
doc = nlp(text)
doc.ents

(Eagle Ridge Project,
 Iron Valley,
 Eagle Ridge deposit,
 Eagle Ridge,
 Eagle Ridge Project)

In [1]:
from project_name_tagger.train_ner import make_spacy_dataset

dataset = make_spacy_dataset("data.csv")

  0%|          | 0/8000 [00:00<?, ?it/s]

In [5]:
import importlib
from project_name_tagger import train_ner
importlib.reload(train_ner)

<module 'project_name_tagger.train_ner' from '/home/ravi/work/mining/project/project_name_tagger/train_ner.py'>

In [11]:
from project_name_tagger.train_ner import to_jsonl

dataset = [x for x in dataset if x[0] != '']


In [29]:
import spacy
from spacy.tokens import DocBin

def to_spacy(data, out):
    nlp = spacy.blank("en")
    doc_bin = DocBin()
    
    for text, entities in data:
        doc = nlp.make_doc(text)
        ents = []
        for start, end, label in entities:
            span = doc.char_span(start, end, label=label)
            if span is not None:
                ents.append(span)
        doc.ents = ents
        doc_bin.add(doc)
    
    doc_bin.to_disk(out)


In [51]:
with_p = [s for s in dataset if len(s[1]) > 0]
without_p = [s for s in dataset if len(s[1]) == 0]

In [52]:
random.shuffle(without_p)

In [53]:
balanced = with_p + without_p[:len(with_p)]

In [54]:
import random
dev_ratio = 0.2
random.shuffle(balanced)
split = int(len(balanced) * (1 - dev_ratio))
train_data = balanced[:split]
dev_data = balanced[split:]

to_jsonl(train_data, "project_name_tagger/train.jsonl")
to_jsonl(dev_data, "project_name_tagger/dev.jsonl")

In [57]:
to_spacy(train_data, 'project_name_tagger/train.spacy')

In [59]:
balanced

[('(5)\xa0 \xa0\xa0Mineral Reserves may be subject to legal, political, environmental and other risks and uncertainties',
  []),
 ('Early warning reports in respect of the foregoing will be filed by Agnico Eagle in accordance with applicable securities laws',
  [(67, 79, 'PROJECT')]),
 ('support the mining sector;', []),
 ('Canada Nickel provides investors with leverage to nickel in low political risk jurisdictions',
  [(0, 13, 'PROJECT')]),
 ('("RNR"), owner of the Aguablanca Ni-Cu Project in Spain, has entered into an agreement with SCP Resource Finance LP and ECM Capital Advisors Ltd',
  [(2, 5, 'PROJECT')]),
 ('Nuclear Fuels logo (CNW Group/Nuclear Fuels Inc.)', [(20, 48, 'PROJECT')]),
 ('While Decisive and its subsidiaries will be impacted if significant long-term blanket tariffs are imposed on Canadian goods with respect to its U.S',
  [(6, 14, 'PROJECT')]),
 ('LUCA ACHIEVES TARGETED THROUGHPUT RATES AT CAMPO MORADO AND TAHUEHUTO MINES',
  [(43, 55, 'PROJECT')]),
 ('SOURCE Clifto

In [58]:
to_spacy(dev_data, 'project_name_tagger/dev.spacy')

In [2]:
import spacy

nlp = spacy.load('project_name_tagger/trf_model/model-best/')

ValueError: [E002] Can't find factory for 'transformer' for language English (en). This usually happens when spaCy calls `nlp.create_pipe` with a custom component name that's not registered on the current language class. If you're using a custom component, make sure you've added the decorator `@Language.component` (for function components) or `@Language.factory` (for class components).

Available factories: merge_noun_chunks, merge_entities, merge_subtokens, en.lemmatizer

In [3]:
!pip show spacy-transformers